# CC-Bench trajectories preview notebook

This notebook loads the public **zai-org/CC-Bench-trajectories** dataset from Hugging Face and previews the first examples in a compact, readable format.

**Source**  
The dataset card says the dataset can be loaded with `datasets.load_dataset("zai-org/CC-Bench-trajectories")`. It contains complete coding-agent trajectories with fields such as `trajectory`, `model_name`, `task_category`, token counts, and tool-call statistics. citeturn836420view0

In [ ]:
!pip -q install datasets pandas

In [1]:
import json
import ast
from pathlib import Path
from typing import Any
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 200)

def truncate(text: Any, n: int = 300) -> str:
    if text is None:
        return ""
    s = str(text)
    return s if len(s) <= n else s[:n] + " …"

def maybe_parse_json(x: Any):
    if isinstance(x, (dict, list)):
        return x
    if not isinstance(x, str):
        return x
    x = x.strip()
    if not x:
        return x
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(x)
        except Exception:
            pass
    return x

def first_dialogue_excerpt(obj: Any, max_items: int = 4) -> str:
    parsed = maybe_parse_json(obj)
    if isinstance(parsed, list):
        parts = []
        for item in parsed[:max_items]:
            if isinstance(item, dict):
                role = item.get("role") or item.get("userType") or item.get("type") or "item"
                content = item.get("content") or item.get("text") or item.get("message") or item.get("action") or item.get("thought") or item
                parts.append(f"{role}: {truncate(content, 180)}")
            else:
                parts.append(truncate(item, 180))
        return "\n".join(parts)
    if isinstance(parsed, dict):
        return truncate(json.dumps(parsed, indent=2), 500)
    return truncate(parsed, 500)

def choose_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

In [2]:
from datasets import load_dataset

ds = load_dataset("zai-org/CC-Bench-trajectories")
train_df = ds["train"].to_pandas()
print(ds)
print(train_df.shape)
display(train_df.head())

DatasetDict({
    train: Dataset({
        features: ['id', 'task_id', 'trajectory', 'model_name', 'task_category', 'user_messages', 'assistant_messages', 'total_input_tokens', 'total_output_tokens', 'total_tokens', 'tool_calls', 'tool_failures', 'failure_rate'],
        num_rows: 370
    })
})
(370, 13)


,id,task_id,trajectory,model_name,task_category,user_messages,assistant_messages,total_input_tokens,total_output_tokens,total_tokens,tool_calls,tool_failures,failure_rate
0,2,1,"[{""parentUuid"": null, ""isSidechain"": false, ""userType"": ""external"", ""cwd"": ""/app/projects"", ""sessionId"": ""2177b41b-6296-42b6-b5d3-ebb19f51d183"", ""version"": ""1.0.58"", ""gitBranch"": """", ""type"": ""user...",DeepSeek-V3.1-Terminus,frontend_development,1,6,497,111,608,4,0,0.0
1,3,1,"[{""parentUuid"": null, ""isSidechain"": false, ""userType"": ""external"", ""cwd"": ""/app/projects"", ""sessionId"": ""d15e5058-4a33-4ddb-a737-fa03f88104da"", ""version"": ""1.0.58"", ""gitBranch"": """", ""type"": ""user...",GLM-4.5,frontend_development,1,6,493,175,668,4,0,0.0
2,4,1,"[{""parentUuid"": null, ""isSidechain"": false, ""userType"": ""external"", ""cwd"": ""/app/projects"", ""sessionId"": ""10af0e53-1430-44b8-83e6-46d55562c084"", ""version"": ""1.0.58"", ""gitBranch"": """", ""type"": ""user...",Kimi-K2-0905,frontend_development,1,6,553,204,757,4,0,0.0
3,5,1,"[{""parentUuid"": null, ""isSidechain"": false, ""userType"": ""external"", ""cwd"": ""/app/projects"", ""sessionId"": ""8eac0536-bbc4-41ff-9588-431213c9cfea"", ""version"": ""1.0.58"", ""gitBranch"": """", ""type"": ""user...",Claude-Sonnet-4,frontend_development,1,5,501,170,671,4,0,0.0
4,6,1,"[{""parentUuid"": null, ""isSidechain"": false, ""userType"": ""external"", ""cwd"": ""/app/projects"", ""sessionId"": ""a3e353f2-a199-4626-b575-bec10c715d0a"", ""version"": ""1.0.128"", ""gitBranch"": """", ""type"": ""use...",GLM-4.6,frontend_development,1,9,843,174,1017,4,0,0.0


In [3]:
overview_cols = [
    c for c in [
        "id", "task_id", "model_name", "task_category",
        "user_messages", "assistant_messages", "tool_calls",
        "tool_failures", "failure_rate", "total_tokens"
    ] if c in train_df.columns
]
preview = train_df[overview_cols].head(10).copy()
display(preview)

,id,task_id,model_name,task_category,user_messages,assistant_messages,tool_calls,tool_failures,failure_rate,total_tokens
0,2,1,DeepSeek-V3.1-Terminus,frontend_development,1,6,4,0,0.0,608
1,3,1,GLM-4.5,frontend_development,1,6,4,0,0.0,668
2,4,1,Kimi-K2-0905,frontend_development,1,6,4,0,0.0,757
3,5,1,Claude-Sonnet-4,frontend_development,1,5,4,0,0.0,671
4,6,1,GLM-4.6,frontend_development,1,9,4,0,0.0,1017
5,8,2,DeepSeek-V3.1-Terminus,frontend_development,1,18,11,0,0.0,4677
6,9,2,GLM-4.5,frontend_development,1,8,6,1,16.7,1196
7,10,2,Kimi-K2-0905,frontend_development,1,5,3,0,0.0,570
8,11,2,Claude-Sonnet-4,frontend_development,1,7,6,1,16.7,1041
9,12,2,GLM-4.6,frontend_development,4,15,7,0,0.0,2460


In [4]:
text_col = choose_first_existing(train_df.columns, ["trajectory"])
example_rows = []
for _, row in train_df.head(5).iterrows():
    example_rows.append({
        "id": row.get("id"),
        "task_id": row.get("task_id"),
        "model_name": row.get("model_name"),
        "task_category": row.get("task_category"),
        "tool_calls": row.get("tool_calls"),
        "tool_failures": row.get("tool_failures"),
        "failure_rate": row.get("failure_rate"),
        "trajectory_excerpt": first_dialogue_excerpt(row.get(text_col)),
    })
example_df = pd.DataFrame(example_rows)
display(example_df)

,id,task_id,model_name,task_category,tool_calls,tool_failures,failure_rate,trajectory_excerpt
0,2,1,DeepSeek-V3.1-Terminus,frontend_development,4,0,0.0,"external: {'role': 'user', 'content': '请用 HTML、JavaScript 完成15x15 的五子棋游戏，游戏胜利后显示赢家并放一个彩蛋'}\nexternal: {'id': 'add43d59-5181-4e5c-89a7-d7e9689698fd', 'type': 'message', 'role': 'assistant', 'model'..."
1,3,1,GLM-4.5,frontend_development,4,0,0.0,"external: {'role': 'user', 'content': '请用 HTML、JavaScript 完成15x15 的五子棋游戏，游戏胜利后显示赢家并放一个彩蛋'}\nexternal: {'id': 'msg_20250924134850ee3e367794fe439d', 'type': 'message', 'role': 'assistant', 'model': ..."
2,4,1,Kimi-K2-0905,frontend_development,4,0,0.0,"external: {'role': 'user', 'content': '请用 HTML、JavaScript 完成15x15 的五子棋游戏，游戏胜利后显示赢家并放一个彩蛋'}\nexternal: {'id': 'msg_chatcmpl-68d5f7acd143e44070e05f44', 'type': 'message', 'role': 'assistant', 'conte..."
3,5,1,Claude-Sonnet-4,frontend_development,4,0,0.0,"external: {'role': 'user', 'content': '请用 HTML、JavaScript 完成15x15 的五子棋游戏，游戏胜利后显示赢家并放一个彩蛋'}\nexternal: {'id': 'msg_01NXcFQMsbDbKxhqJXzzUSCR', 'type': 'message', 'role': 'assistant', 'model': 'claud..."
4,6,1,GLM-4.6,frontend_development,4,0,0.0,"external: {'role': 'user', 'content': '请用 HTML、JavaScript 完成15x15 的五子棋游戏，游戏胜利后显示赢家并放一个彩蛋'}\nexternal: {'id': 'msg_a089c3664cf544a591d5a3d0', 'type': 'message', 'role': 'assistant', 'model': 'claud..."


In [5]:
if "task_category" in train_df.columns:
    display(train_df["task_category"].value_counts().rename_axis("task_category").reset_index(name="count"))
if "model_name" in train_df.columns:
    display(train_df["model_name"].value_counts().rename_axis("model_name").reset_index(name="count"))

,task_category,count
0,application_development,135
1,frontend_development,85
2,ui_optimization,65
3,machine_learning,40
4,data_analysis,25
5,build_deployment,20


,model_name,count
0,DeepSeek-V3.1-Terminus,74
1,GLM-4.5,74
2,Kimi-K2-0905,74
3,Claude-Sonnet-4,74
4,GLM-4.6,74
